# Spatial Autocorrelation

> Although it may be viewed as a special case of correlation, spatial autocorrelation has a meaning all its own. Whereas correlation statistics were designed to show relationships between variables, autocorrelation statistics are designed to show correlations within variables, and spatial autocorrelation shows the correlation within variables across space.
>
> Arthur Getis (2007)

Among the most important concepts in spatial analysis is spatial dependence: the idea that a value measured at one location, such as a home price, an infection rate, or job growth, is related to the values nearby. Dependence in a measured outcome can come from measurement error, from a spatial spillover, or from some other spatially correlated process, and since causality is rarely provable, we usually study the association between measurements in space. Measures of spatial association answer questions like these:

- Is there some spatial process at play, or is the pattern of values random in space?
- At what scale do spatial relationships operate? Do the same intuitions hold when we change the notion of neighborhood in the graph?
- Are there stronger patterns in some parts of the map, such as clusters, hot spots, and cold spots?

The answers can generate new hypotheses, which is why the literature speaks of exploratory spatial data analysis. They are also diagnostics, for example when we ask whether a regression's residuals are random in space, a question the afternoon takes up.

The absence of spatial association, a map pattern with nothing to detect, is called complete spatial randomness (CSR), and it is the null hypothesis for the statistics below. Rather than assume a statistic follows a known distribution under CSR, modern spatial analysis simulates it. Random permutations shuffle the observed values across locations thousands of times, the statistic is computed for each shuffle, and the observed statistic is compared with that reference distribution. The share of permutations at least as extreme as the observed value is a pseudo $p$-value.

In [ ]:
import esda
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from libpysal.graph import Graph

## Home values in San Diego

We examine the median home value in the census tracts of San Diego County.

In [ ]:
scag = gpd.read_parquet("data/scag_region.parquet")
san_diego = scag[scag.geoid.str[:5] == "06073"].to_crs(26911).set_index("geoid")
san_diego = san_diego.dropna(subset=["median_home_value"])
san_diego.shape

In [ ]:
san_diego["median_home_value"].hist(bins=30);

In [ ]:
ax = san_diego.plot(
    "median_home_value", scheme="quantiles", k=5, cmap="GnBu", edgecolor="white",
    linewidth=0.1, legend=True, legend_kwds={"loc": "upper left", "bbox_to_anchor": (1, 1)}, figsize=(8, 8),
)
ax.set_title("Median home value (quintiles)")
ax.set_axis_off()

If home values were distributed at random in space, we would see no clustering of similar values on the map. Our eyes are drawn instead to the dark clusters along the coast and the lighter ones in the south and east. But our brains are powerful pattern detectors, sometimes too powerful: they find patterns where there are none, a particular hazard with irregular polygons of very different sizes. The statistics below test what the eye suggests.

## The spatial lag

The question underneath every measure of spatial association is the relationship between each unit and its neighborhood. We define the neighborhood with a queen contiguity graph, row-standardized.

In [ ]:
graph = Graph.build_contiguity(san_diego, rook=False).transform("r")

Treating the graph as a matrix, $W$, gives a measure of the neighborhood around each observation: multiply $W$ by the variable $y$ to get the spatial lag, $Wy$, or

$$\mathrm{lag}_i = \sum_j w_{ij} y_j.$$

With row-standardized weights, the spatial lag is the average of the neighbors' values. A unit is not a member of its own neighborhood, so measuring spatial association means relating $y$ to $Wy$.

In [ ]:
y = san_diego["median_home_value"]
san_diego["lag_home_value"] = graph.lag(y)

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(12, 6))
for axis, column, title in zip(
    ax, ["median_home_value", "lag_home_value"], ["Median home value", "Spatial lag"]
):
    san_diego.plot(column, scheme="quantiles", k=5, cmap="GnBu", edgecolor="white", linewidth=0.1, ax=axis)
    axis.set_title(title)
    axis.set_axis_off()

The map of the spatial lag enhances the impression of similar values near one another; in effect, it is a local smoother. Associating each tract's value with its neighborhood's value by eye is still hard, which is where the statistics come in.

## Global spatial autocorrelation

### Binary outcomes: join counts

The join count statistic measures spatial autocorrelation in a binary variable. A join is a pair of neighbors, an edge in the graph. If each unit takes one of two values, black or white, as on a chessboard,

In [ ]:
image = np.zeros(81)
image[::2] = 1
plt.matshow(image.reshape(9, 9), cmap="Greys")
plt.axis("off");

then a join between neighbors is one of three kinds: black-black (BB), white-white (WW), or black-white (BW). The join count statistic compares the joins we observe with the joins we would expect under complete spatial randomness. In the language of social networks, it measures homophily: rejecting the null means that observations in the same category are more connected to one another than chance would make them, and in a spatial graph, more connected means closer in space.

We make home values binary by splitting at the median, so that black tracts are those above it.

In [ ]:
yb = (y > y.median()).astype(int)
yb.sum()

In [ ]:
ax = san_diego.assign(above_median=yb.map({0: "below", 1: "above"})).plot(
    "above_median", cmap="binary", edgecolor="grey", linewidth=0.2, legend=True, figsize=(8, 8)
)
ax.set_axis_off()

Given 313 black tracts, how many black-black joins would we expect if they were placed on the map at random?

In [ ]:
np.random.seed(12345)
jc = esda.Join_Counts(yb, graph)
jc.bb, jc.ww, jc.bw

The three kinds of join exhaust the possibilities, so they sum to the number of unique neighbor pairs in the graph.

In [ ]:
jc.bb + jc.ww + jc.bw, graph.n_edges / 2

To judge whether 754 black-black joins is unusual, `esda` permutes the values across the map 999 times and counts the joins in each permutation. The average count under complete spatial randomness is

In [ ]:
jc.mean_bb

In [ ]:
sns.kdeplot(jc.sim_bb, fill=True)
plt.axvline(jc.bb, color="r")
plt.axvline(jc.mean_bb, color="k")
plt.xlabel("BB counts");

The density is the reference distribution of black-black counts, the black line its mean, and the red line the count we observe, far out in the upper tail. The pseudo $p$-value says the same thing.

In [ ]:
jc.p_sim_bb

None of the 999 random maps produced as many black-black joins as the observed one, so we reject complete spatial randomness: tracts with high home values neighbor one another far more often than chance would allow.

### Continuous outcomes: Moran's I

Splitting at the median threw away most of the information in home values. Moran's $I$ measures spatial autocorrelation in the continuous variable.

In [ ]:
np.random.seed(12345)
mi = esda.Moran(y, graph)
mi.I, mi.p_sim

In [ ]:
f, ax = plt.subplots(1, 2, figsize=(12, 4))
mi.plot_simulation(ax=ax[0])
mi.plot_scatter(ax=ax[1]);

The left panel shows the reference distribution of $I$ under random permutations and the observed value in red, well outside it. The right panel is the Moran scatterplot: each tract's standardized home value on the horizontal axis and the average of its neighbors on the vertical. Moran's $I$ is the slope of the line through it. Home values in a tract are strongly related to home values nearby, and the relationship cannot be attributed to chance.

## Local spatial autocorrelation

A global statistic summarizes the whole map in one number. Local indicators of spatial association (LISA) apply the same logic to each observation, identifying where the clustering is: groups of tracts whose values are statistically higher or lower than their surroundings.

In [ ]:
np.random.seed(12345)
li = esda.Moran_Local(y, graph)

Instead of a single statistic, `li.Is` holds a local $I_i$ for every tract, and `li.p_sim` a pseudo $p$-value for each. Here the permutations are conditional: each tract keeps its own value while its neighbors' values are shuffled, which builds a separate reference distribution for every location.

In [ ]:
(li.p_sim < 0.05).sum()

### Hot spots, cold spots, and spatial outliers

Each significant tract falls in one of the four quadrants of the Moran scatterplot:

- High-High: a high value among high values, a hot spot.
- Low-Low: a low value among low values, a cold spot.
- High-Low: a high value among low values, a spatial outlier sometimes called a diamond in the rough.
- Low-High: a low value among high values, a spatial outlier sometimes called a doughnut.

In [ ]:
ax = li.plot_scatter(crit_value=0.05)
ax.set_xlabel("Median home value")
ax.set_ylabel("Spatial lag of median home value");

In [ ]:
ax = li.plot(san_diego, crit_value=0.05, figsize=(8, 8))
ax.set_axis_off()

In [ ]:
li.explore(san_diego, crit_value=0.05)

`explore` draws the same map on an interactive web map, so we can pan, zoom, and hover over tracts to see where the clusters are.

In [ ]:
san_diego["cluster"] = li.get_cluster_labels(crit_value=0.05)
san_diego.groupby("cluster")["median_home_value"].agg(["count", "median"]).round(0)

Home values in San Diego sort into large, contiguous hot and cold spots, with almost no outliers: no high-value tract stands alone among low-value neighbors, and only a handful of low-value tracts sit inside high-value areas. For a housing market, that is what we should expect, since neighboring homes share schools, amenities, and access, and their prices are set partly by comparison with one another. It is also the pattern that makes a spatial model necessary, because observations this dependent violate the independence that an ordinary regression assumes.

<div class="alert alert-success" style="font-size:120%">
<b>Exercise</b>: <br>
Repeat the global and local analysis for median household income, <code>median_household_income</code>. Is income more or less clustered than home values, and do its hot and cold spots fall in the same places?
</div>

In [ ]:
# %load solutions/03.py

---

<a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc-sa/4.0/88x31.png" /></a><br /><span xmlns:dct="http://purl.org/dc/terms/" property="dct:title">Spatial Autocorrelation</span> by Serge Rey and Eli Knaap is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License</a>.